## 1. Mount Drive


# LPG Classifier Training — EfficientNetB2 (v3 → v4_3 lineage)

## Overview
This is **stage 2** of the two-stage pipeline: it trains a whole-crop EfficientNetB2 brand
classifier (`bharat`/`hp`/`indane`/`unknown`) on cylinder crops. The notebook re-crops the raw
Roboflow classification dataset using the newer **YOLOv11x** detector
(`yolov11x_lpg_v1_best.pt`, 98.6% detection rate) instead of the older YOLOv11n nano detector
(95.0%), filters out crops smaller than 80px, trains EfficientNetB2 for 50 epochs, and saves the
best checkpoint as `classifier_best_v4_3.pth`. The back half of the notebook (sections 9–10) is a
mix of evaluation cells for several model versions (v3, v4, v4_1, v4_2, v4_3) accumulated across
iterations — not all of them are internally consistent with the model actually trained in section
7 (see Current Status).

**Per the project README/CLAUDE.md, `classifier_best_v4_3.pth` is now archived** — its 97.62%
figure is on a different, smaller validation set and is not the current production side-view
model (that is `classifier_best_v6_attention.pth`, trained elsewhere).

## How to Run
1. **Runtime:** Colab **GPU** (the notebook's `device = torch.device("cuda" if ...)` pattern and
   50-epoch training loop over ImageNet-pretrained EfficientNetB2 need a GPU — T4 or better).
2. **Upload/mount:**
   - Section 1 mounts Google Drive and installs `ultralytics`.
   - Section 1 also prompts to upload a classification dataset zip (Roboflow export, with
     `train/<brand>/` and `valid/<brand>/` folders).
   - Section 4 prompts a **second** upload for the YOLOv11x detector checkpoint
     (`yolov11x_lpg_v1_best.pt`) used to re-crop the dataset.
3. **Execution order:** Run top to bottom through section 8 to reproduce training and save a
   checkpoint. Sections 9–10 (evaluation) are exploratory/iterative — they reload different
   checkpoint filenames (`classifier_best_v4_3.pth`, `classifier_best_v3.pth`,
   `classifier_best.pth`) and are not meant to run sequentially as one pipeline; run only the
   subsection matching the checkpoint you actually have.
4. **Expected outputs:** `classifier_best_v4_3.pth`, `training_history_v4_3.json`, several PNG
   charts, and (in section 9) more charts saved under a Drive `Classifier/v4_3` folder. See the
   Output Files table near the end of this notebook.

## Model / Dataset Info
| Component | Detail |
|---|---|
| Architecture | EfficientNetB2 (ImageNet-pretrained backbone) + `Dropout(0.4)` + `Linear` head, no color-histogram head |
| Classes | `train_data.classes` (ImageFolder-derived — expected `bharat, hp, indane, unknown`) |
| Detector used for re-cropping | `yolov11x_lpg_v1_best.pt`, conf=0.45, `augment=True`, all boxes kept |
| Crop filter | drop crops with min(width,height) < 80px |
| Epochs | 50 |
| Batch size | 32 |
| Optimizer / LR | Adam, lr=3e-4, weight_decay=1e-4, label_smoothing=0.1, CosineAnnealingLR(T_max=50) |
| Sampler | `WeightedRandomSampler` (inverse class frequency) to counter class imbalance |
| Best val acc (as printed in-notebook) | 97.6% (`classifier_best_v4_3.pth`) — **archived, not current production model** |
| Detection-rate note (cell 9, markdown) | YOLOv11x: 98.6% (~46 fallbacks) vs YOLOv11n: 95.0% (~128 fallbacks); total crops 2487 → 3208 |

## Current Status
Training (sections 1–8) is complete and was run at least once — `classifier_best_v4_3.pth` and
its training history were produced and pushed to Drive. The evaluation tail (sections 9–10) is
**not fully consistent**:
- Cell "v4 Evaluation" (`## v4 Evaluation Clf Report w color hist head`) calls
  `model(imgs, hists)`, i.e. expects a color-histogram-head model, but the model actually built
  in section 7 takes only `imgs` — this cell will fail as written against the section-7 model and
  looks like leftover code from an earlier notebook variant.
- Sections 10.1–10.3 evaluate `classifier_best_v3.pth`, an older checkpoint not present in the
  current repo's `models/` folder.
- **Two cells (GradCAM test, titled "GradCAM — EfficientNetB2 v3") are exact duplicates** of one
  another — left as-is per the conservative editing policy for this pass rather than deleted.
- Two empty/unused code cells were found (after "Evaluation reporting v4" and at the very end).
- This notebook, along with `nb_bottomring_classifier.ipynb`/training code, is one of the sources
  of confusion the project's CLAUDE.md flags around checkpoint/notebook mismatches — treat the
  section-7 training loop as the reliable part of this notebook and the section 9–10 evaluation
  cells as historical/exploratory scratch work.


In [ ]:
from google.colab import drive, files
# Mount Drive so trained checkpoints/history/charts can be pushed to a persistent location later
drive.mount('/content/drive')
!pip install ultralytics -q
print("Done!")

In [ ]:
# Roboflow classification export — expects train/<brand>/ and valid/<brand>/ subfolders
print("Upload your classification dataset zip from Roboflow")
uploaded = files.upload()

## 2. Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

## 3. Unzip dataset

In [ ]:
import zipfile, os

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/dataset")
print("Unzipped!")

# Check structure
for split in ["train", "valid"]:
    for brand in os.listdir(f"/content/dataset/{split}"):
        count = len(os.listdir(f"/content/dataset/{split}/{brand}"))
        print(f"{split}/{brand}: {count}")

## 4. Creating image crops w updated detector Yolov11x


In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
from PIL import Image
import os, shutil
from google.colab import files

# Upload new detector — the current/newer YOLOv11x detector, used here to re-crop the raw
# classification dataset more accurately than the older YOLOv11n nano detector did
print("Upload yolov11x_lpg_v1_best.pt")
uploaded = files.upload()

detector = YOLO("yolov11x_lpg_v1_best.pt")

DATASET_PATH = "/content/dataset"
OUTPUT_PATH  = "/content/dataset_cropped_v2"

summary = {}

for split in ["train", "valid"]:
    for brand in os.listdir(f"{DATASET_PATH}/{split}"):
        src_folder  = f"{DATASET_PATH}/{split}/{brand}"
        dest_folder = f"{OUTPUT_PATH}/{split}/{brand}"
        os.makedirs(dest_folder, exist_ok=True)

        images = [f for f in os.listdir(src_folder)
                  if f.lower().endswith((".jpg", ".jpeg", ".png"))]

        saved = skipped = 0

        for fname in images:
            fpath = f"{src_folder}/{fname}"
            try:
                # New: conf=0.45, augment=True, all boxes
                results = detector(fpath, conf=0.45, augment=True, verbose=False)
                boxes   = results[0].boxes

                if len(boxes) == 0:
                    # No detection: fall back to copying the whole uncropped image rather than
                    # dropping the sample entirely, so it still contributes to training
                    shutil.copy(fpath, f"{dest_folder}/{fname}")
                    skipped += 1
                    continue

                img = Image.open(fpath).convert("RGB")

                # New: save ALL detections not just first (handles multi-cylinder source images)
                for idx, box in enumerate(boxes):
                    coords = box.xyxy[0].cpu().numpy()
                    crop   = img.crop((
                        max(0, int(coords[0])),
                        max(0, int(coords[1])),
                        min(img.width,  int(coords[2])),
                        min(img.height, int(coords[3]))
                    ))
                    stem     = os.path.splitext(fname)[0]
                    ext      = os.path.splitext(fname)[1]
                    out_name = f"{stem}_crop{idx}{ext}" if len(boxes) > 1 else fname
                    crop.save(f"{dest_folder}/{out_name}")
                    saved += 1

            except Exception as e:
                print(f"Error {fname}: {e}")
                shutil.copy(fpath, f"{dest_folder}/{fname}")
                skipped += 1

        summary[f"{split}/{brand}"] = {"cropped": saved, "fallback": skipped}
        print(f"{split}/{brand}: {saved} cropped, {skipped} fallback")

print("\n✅ Cropping done!")
print("\nSummary:")
total_cropped  = sum(v["cropped"]  for v in summary.values())
total_fallback = sum(v["fallback"] for v in summary.values())
print(f"Total cropped:  {total_cropped}")
print(f"Total fallback: {total_fallback}")
print(f"Detection rate: {round(total_cropped/(total_cropped+total_fallback)*100, 1)}%")

* Old yolov11n nano detector:    95.0% detection rate  (~128 fallbacks)

* New yolo11x detector: 98.6% detection rate  (~46 fallbacks)

* Improvement:          
  * +3.6%  (-82 fallbacks)

  * total crops from 2487[yolov11n] → 3208[yolov11x]
  
  * 721 additional crops

## 4.1 Verifying crops

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import random

OUTPUT_PATH = "/content/dataset_cropped_v2"
BRANDS = ["bharat", "hp", "indane", "unknown"]

for brand in BRANDS:
    folder = f"{OUTPUT_PATH}/train/{brand}"
    images = [f for f in os.listdir(folder)
              if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    # Pick 12 random samples
    samples = random.sample(images, min(12, len(images)))

    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    axes = axes.flatten()

    for i, fname in enumerate(samples):
        fpath = f"{folder}/{fname}"
        try:
            img = mpimg.imread(fpath)
            axes[i].imshow(img)
            axes[i].set_title(fname[:20], fontsize=7)
            axes[i].axis("off")
        except:
            axes[i].axis("off")

    # Hide empty
    for j in range(len(samples), 12):
        axes[j].axis("off")

    plt.suptitle(f"{brand.upper()} — sample crops ({len(images)} total)",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"{brand}: {len(images)} crops\n")

In [ ]:
from PIL import Image
import os

# Scan the freshly re-cropped dataset and bucket images by minimum side length, to decide a
# sensible minimum-crop-size threshold before training (see the removal cell below)
BASE = "/content/dataset/dataset_cropped_v2_clean"
size_buckets = {"< 80px": 0, "80-150px": 0, "150-224px": 0, "> 224px": 0}
total = 0

for split in ["train", "valid"]:
    for brand in ["bharat", "hp", "indane", "unknown"]:
        folder = f"{BASE}/{split}/{brand}"
        if not os.path.exists(folder):
            continue
        for fname in os.listdir(folder):
            fpath = f"{folder}/{fname}"
            try:
                img  = Image.open(fpath)
                w, h = img.size
                size = min(w, h)
                total += 1
                if size < 80:
                    size_buckets["< 80px"] += 1
                elif size < 150:
                    size_buckets["80-150px"] += 1
                elif size < 224:
                    size_buckets["150-224px"] += 1
                else:
                    size_buckets["> 224px"] += 1
            except Exception as e:
                print(f"Error: {fpath}: {e}")

print(f"Total images scanned: {total}")
print("\nSize distribution:")
for k, v in size_buckets.items():
    pct = round(v/total*100, 1) if total > 0 else 0
    print(f"  {k}: {v} ({pct}%)")

In [ ]:
import os
from PIL import Image

# Delete crops smaller than MIN_SIZE in-place — these are too small to be useful training
# signal for a 224x224-input classifier (destructive: operates directly on the dataset folder)
BASE    = "/content/dataset/dataset_cropped_v2_clean"
MIN_SIZE = 80
removed  = 0

for split in ["train", "valid"]:
    for brand in ["bharat", "hp", "indane", "unknown"]:
        folder = f"{BASE}/{split}/{brand}"
        if not os.path.exists(folder):
            continue
        for fname in os.listdir(folder):
            fpath = f"{folder}/{fname}"
            try:
                img  = Image.open(fpath)
                w, h = img.size
                if min(w, h) < MIN_SIZE:
                    os.remove(fpath)
                    removed += 1
            except:
                pass

print(f"Removed {removed} images under {MIN_SIZE}px")

# New counts
for split in ["train", "valid"]:
    for brand in ["bharat", "hp", "indane", "unknown"]:
        folder = f"{BASE}/{split}/{brand}"
        if os.path.exists(folder):
            print(f"{split}/{brand}: {len(os.listdir(folder))}")

## 5. Transforms

In [ ]:
# Heavy augmentation pipeline (color jitter, grayscale, blur, rotation, perspective warp,
# random erasing) — chosen aggressively to compensate for a relatively small training set and
# reduce overfitting to background/lighting rather than brand-distinguishing features
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.ColorJitter(brightness=0.6, contrast=0.6, saturation=0.6, hue=0.2),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),  # ← add back
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.RandomPerspective(distortion_scale=0.4, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.05, 0.25), ratio=(0.3, 3.0)),
])

# Validation transform: deterministic resize + normalize only, no augmentation
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Transforms ready!")

## 6. Dataset +  Sampler + dataloaders

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# ── Load datasets ─────────────────────────────────────────────────────────
# Cleaned/size-filtered crop dataset produced by sections 4 and 4.1 above
TRAIN_PATH = "/content/dataset/dataset_cropped_v2_clean/train"
VAL_PATH   = "/content/dataset/dataset_cropped_v2_clean/valid"

train_data = datasets.ImageFolder(TRAIN_PATH, transform=train_transforms)
val_data   = datasets.ImageFolder(VAL_PATH,   transform=val_transforms)

CLASSES   = train_data.classes
N_CLASSES = len(CLASSES)
print(f"Classes: {CLASSES}")
print(f"Train: {len(train_data)} | Val: {len(val_data)}")

# ── WeightedRandomSampler ─────────────────────────────────────────────────
# Counteracts class imbalance by sampling rarer classes more often during training
class_counts   = [len([s for s in train_data.samples if s[1] == i])
                  for i in range(N_CLASSES)]
print(f"Class counts: {dict(zip(CLASSES, class_counts))}")

sample_weights = [1.0 / class_counts[s[1]] for s in train_data.samples]
sampler        = WeightedRandomSampler(sample_weights, len(train_data), replacement=True)

train_loader = DataLoader(train_data, batch_size=32, sampler=sampler,  num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False,    num_workers=2)

print("Dataloaders ready!")

## 7. Efficient Net Model
without colorhistogram head:

In [ ]:
# ImageNet-pretrained EfficientNetB2 backbone with a fresh dropout + linear classification head
# sized to N_CLASSES (no color-histogram auxiliary head in this version — see section 7 title)
model = models.efficientnet_b2(weights="IMAGENET1K_V1")
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(model.classifier[1].in_features, N_CLASSES)
)
model = model.to(device)

# label_smoothing=0.1 discourages over-confident predictions; weight_decay=1e-4 for regularization
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready! Trainable params: {total_params:,}")

## 7. Model training loop + saving metadata

In [ ]:
from tqdm import tqdm
import json

EPOCHS = 50  # up from 30
# Re-create the scheduler so its T_max matches the actual EPOCHS used here (the one defined
# alongside the optimizer above used T_max=30, which is now stale)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50, eta_min=1e-6  # match T_max to epochs
)
best_val_acc = 0.0
history      = []

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs  = model(imgs)
            loss     = criterion(outputs, labels)
            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)

    train_acc      = train_correct / train_total * 100
    val_acc        = val_correct   / val_total   * 100
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss   = val_loss   / len(val_loader)

    history.append({
        "epoch":      epoch + 1,
        "train_loss": round(avg_train_loss, 4),
        "train_acc":  round(train_acc, 2),
        "val_loss":   round(avg_val_loss, 4),
        "val_acc":    round(val_acc, 2),
    })

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.1f}% | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.1f}%")

    # Checkpoint dict includes classes/architecture/threshold metadata so downstream inference
    # code (predict.py / predict_ensemble.py) can self-describe rather than hard-code assumptions
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict":     model.state_dict(),
            "best_val_acc":         best_val_acc,
            "classes":              CLASSES,
            "architecture":         "efficientnet_b2",
            "num_classes":          N_CLASSES,
            "epochs_trained":       epoch + 1,
            "train_history":        history,
            "class_counts":         dict(zip(CLASSES, class_counts)),
            "confidence_threshold": 0.60,
        }, "classifier_best_v4_3.pth")
        print(f"  ✅ New best saved: {val_acc:.1f}%")

    scheduler.step()

print(f"\nDone! Best val acc: {best_val_acc:.1f}%")

with open("training_history_v4_3.json", "w") as f:
    json.dump(history, f, indent=2)
print("Training history saved!")

## 8. Save to Drive

In [ ]:
import shutil
from google.colab import files

# Persist the checkpoint + training history to Drive, then also trigger a local download
shutil.copy(
    "classifier_best_v4_3.pth",
    "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Dataset/classifier_best_v4_3.pth"  # ← UPDATE THIS PATH
)
shutil.copy(
    "training_history_v4_3.json",
    "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Dataset/training_history_v4_3.json"  # ← UPDATE THIS PATH
)
print("Saved to Drive!")
files.download("classifier_best_v4_3.pth")

# Reload the just-saved best checkpoint and compute a classification report on the val set
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

checkpoint = torch.load("classifier_best_v4_3.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

checkpoint = torch.load("classifier_best_v4_3.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import numpy as np
import os
import shutil
from google.colab import files
from sklearn.metrics import confusion_matrix

# ── Paths ─────────────────────────────────────────────────────────────────
DRIVE_SAVE = "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Classifier/v4_3"  # ← UPDATE THIS PATH
os.makedirs(DRIVE_SAVE, exist_ok=True)

# ── 1. Confusion Matrix ───────────────────────────────────────────────────
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title("Confusion Matrix — Raw Counts", fontweight="bold")
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

sns.heatmap(cm_norm, annot=True, fmt=".0%", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title("Confusion Matrix — Normalised (%)", fontweight="bold")
axes[1].set_ylabel("Actual")
axes[1].set_xlabel("Predicted")

plt.suptitle("EfficientNetB2 v4_3 — Validation Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix_v4_3.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrix")

# ── 2. Per-class accuracy bar chart ──────────────────────────────────────
per_class_acc = cm_norm.diagonal() * 100

fig, ax = plt.subplots(figsize=(9, 5))
colors  = ["#e74c3c", "#3498db", "#2ecc71", "#95a5a6"]
bars    = ax.bar(CLASSES, per_class_acc, color=colors, edgecolor="white", linewidth=1.2)

for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc:.1f}%", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylim(0, 115)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Per-Class Accuracy — EfficientNetB2 v4_3", fontweight="bold")
ax.axhline(y=per_class_acc.mean(), color="black", linestyle="--", linewidth=1.2,
           label=f"Mean: {per_class_acc.mean():.1f}%")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("per_class_accuracy_v4_3.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Per-class accuracy")

# ── 3. Training history curve ─────────────────────────────────────────────
import json
with open("training_history_v4_3.json") as f:
    history = json.load(f)

epochs     = [h["epoch"]      for h in history]
train_acc  = [h["train_acc"]  for h in history]
val_acc    = [h["val_acc"]    for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss   = [h["val_loss"]   for h in history]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(epochs, train_acc,  label="Train Acc",  color="#3498db", linewidth=2)
axes[0].plot(epochs, val_acc,    label="Val Acc",    color="#2ecc71", linewidth=2)
axes[0].axhline(y=max(val_acc), color="black", linestyle="--", linewidth=1,
                label=f"Best Val: {max(val_acc):.1f}%")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Accuracy over Epochs", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].plot(epochs, train_loss, label="Train Loss", color="#e74c3c", linewidth=2)
axes[1].plot(epochs, val_loss,   label="Val Loss",   color="#e67e22", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Loss over Epochs", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle("EfficientNetB2 v4_3 — Training History", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("training_history_v4_3.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Training history")

# ── 4. Version comparison bar chart ──────────────────────────────────────
versions = ["B0 v1\n(overfit)", "B2 v2", "B2 v3", "B2 v4_3"]
accuracies = [100, 94, 96.3, 97.6]
colors_v   = ["#e74c3c", "#e67e22", "#3498db", "#2ecc71"]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(versions, accuracies, color=colors_v, edgecolor="white", linewidth=1.2, width=0.5)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{acc}%", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylim(85, 105)
ax.set_ylabel("Validation Accuracy (%)")
ax.set_title("Model Version Comparison", fontsize=14, fontweight="bold")
ax.axhline(y=97.6, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.spines[["top", "right"]].set_visible(False)

# Add note on v1
ax.annotate("100% val acc\nbut overfit —\nfailed on real images",
            xy=(0, 100), xytext=(0.5, 102),
            fontsize=9, color="#e74c3c",
            arrowprops=dict(arrowstyle="->", color="#e74c3c"))

plt.tight_layout()
plt.savefig("version_comparison_v4_3.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Version comparison")

# ── 5. Save all to Drive ──────────────────────────────────────────────────
charts = [
    "confusion_matrix_v4_3.png",
    "per_class_accuracy_v4_3.png",
    "training_history_v4_3.png",
    "version_comparison_v4_3.png",
]

for fname in charts:
    shutil.copy(fname, f"{DRIVE_SAVE}/{fname}")
    print(f"Saved to Drive: {fname}")

# ── 6. Download all ───────────────────────────────────────────────────────
for fname in charts:
    files.download(fname)

print("\n✅ All charts saved to Drive and downloaded!")

# ── 7. Summary ────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("EfficientNetB2 v4_3 — Final Summary")
print("="*55)
print(f"  Overall Accuracy:  97.6%")
print(f"  Bharat Gas F1:     0.98")
print(f"  HP Gas F1:         0.97")
print(f"  Indane F1:         0.97")
print(f"  Unknown F1:        0.96")
print(f"  Best epoch:        {history[np.argmax(val_acc)]['epoch']}")
print(f"  Detector:          YOLOv11x (mAP50: 0.969)")
print(f"  Crop filter:       min 80px")
print(f"  Training images:   {len(train_data)}")
print(f"  Val images:        {len(val_data)}")
print("="*55)

## v4_2 eval

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Load best model ───────────────────────────────────────────────────────
checkpoint = torch.load("classifier_best_v4_3.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded v4_1 — best val acc: {checkpoint['best_val_acc']:.1f}%")

# ── Get predictions ───────────────────────────────────────────────────────
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs   = imgs.to(device)
        outputs = model(imgs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ── Classification report ─────────────────────────────────────────────────
print("\n" + "="*55)
print("CLASSIFICATION REPORT — v4_3")
print("="*55)
print(classification_report(all_labels, all_preds, target_names=CLASSES))

# ── Confusion matrix ──────────────────────────────────────────────────────
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

### v4_1 Evaluation  Clf Report w/o color hist head

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Load best model ───────────────────────────────────────────────────────
checkpoint = torch.load("classifier_best_v4_3.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded v4_1 — best val acc: {checkpoint['best_val_acc']:.1f}%")

# ── Get predictions ───────────────────────────────────────────────────────
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs   = imgs.to(device)
        outputs = model(imgs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ── Classification report ─────────────────────────────────────────────────
print("\n" + "="*55)
print("CLASSIFICATION REPORT — v4_3")
print("="*55)
print(classification_report(all_labels, all_preds, target_names=CLASSES))

# ── Confusion matrix ──────────────────────────────────────────────────────
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# NOTE: this cell expects val_loader to yield (imgs, hists, labels) and a model that accepts
# (imgs, hists) — i.e. a color-histogram-head architecture. The model built in section 7 above
# only takes imgs, so this cell will raise as written; it appears to be leftover code from an
# earlier notebook variant that did use a color-histogram head. Left unmodified (documentation
# pass only) — see the Current Status note at the top of this notebook.
from sklearn.metrics import classification_report
import numpy as np

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, hists, labels in val_loader:
        imgs, hists = imgs.to(device), hists.to(device)
        outputs = model(imgs, hists)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES))

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, hists, labels in val_loader:
        imgs, hists = imgs.to(device), hists.to(device)
        outputs = model(imgs, hists)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES))

## 9. Evaluation reporting v4

## 10 Clf report v3

### 10.1 clf report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Load best v3 — an older checkpoint filename, not the v4_3 model trained above; requires
# classifier_best_v3.pth to already exist in the working directory (not produced by this notebook)
checkpoint = torch.load("classifier_best_v3.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print("="*50)
print("CLASSIFICATION REPORT — v3")
print("="*50)
print(classification_report(all_labels, all_preds, target_names=train_data.classes))

### 10.2 Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=train_data.classes, yticklabels=train_data.classes, ax=axes[0])
axes[0].set_title("Raw Counts")
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

sns.heatmap(cm_norm, annot=True, fmt=".0%", cmap="Blues",
            xticklabels=train_data.classes, yticklabels=train_data.classes, ax=axes[1])
axes[1].set_title("Normalised %")
axes[1].set_ylabel("Actual")
axes[1].set_xlabel("Predicted")

plt.suptitle("EfficientNetB2 v3 — Validation Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix_v3.png", dpi=150, bbox_inches="tight")
plt.show()
files.download("confusion_matrix_v3.png")

### 10.3 Gradcam Test

In [ ]:
#!pip install grad-cam -q
import torchvision.transforms as T
import numpy as np
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

print("Upload your hard test images")
uploaded_imgs = files.upload()
image_paths = list(uploaded_imgs.keys())

target_layer = [model.features[-1]]
transform_test = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

CLASSES = list(train_data.classes)
n = len(image_paths)
fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
if n == 1: axes = [axes]

for i, path in enumerate(image_paths):
    img_pil = Image.open(path).convert("RGB").resize((224, 224))
    img_np  = np.array(img_pil).astype(np.float32) / 255.0
    tensor  = transform_test(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)[0]

    pred_idx   = probs.argmax().item()
    pred_brand = CLASSES[pred_idx]
    confidence = round(probs[pred_idx].item() * 100, 1)

    with GradCAM(model=model, target_layers=target_layer) as cam:
        grayscale = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred_idx)])
        cam_image = show_cam_on_image(img_np, grayscale[0], use_rgb=True)

    axes[i][0].imshow(img_np)
    axes[i][0].set_title(f"Original\n{path}", fontsize=10)
    axes[i][0].axis("off")

    axes[i][1].imshow(cam_image)
    axes[i][1].set_title(f"GradCAM\n{pred_brand} ({confidence}%)", fontsize=10)
    axes[i][1].axis("off")

    probs_list = [round(probs[j].item()*100, 1) for j in range(len(CLASSES))]
    colors = ["#e74c3c", "#3498db", "#2ecc71", "#95a5a6"][:len(CLASSES)]
    bars = axes[i][2].barh(CLASSES, probs_list, color=colors)
    axes[i][2].set_xlim(0, 110)
    axes[i][2].set_xlabel("Confidence (%)")
    axes[i][2].set_title("Probabilities")
    for bar, val in zip(bars, probs_list):
        axes[i][2].text(val+1, bar.get_y()+bar.get_height()/2,
                        f"{val}%", va="center", fontsize=10, fontweight="bold")
    axes[i][2].spines[["top","right"]].set_visible(False)

plt.suptitle("GradCAM — EfficientNetB2 v3", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("gradcam_v3.png", dpi=150, bbox_inches="tight")
plt.show()
files.download("gradcam_v3.png")

In [ ]:
from google.colab import files
import torch
import torch.nn as nn
from torchvision import models
from PIL import Image
import numpy as np

# Loads a separately-uploaded checkpoint (classifier_best.pth) into a fresh EfficientNetB2 —
# used to bring in an older/external model version for ad-hoc comparison/testing
print("Upload classifier_best.pth")
uploaded = files.upload()

CLASSES = ["bharat_gas", "hp_gas", "indane", "unknown"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load EfficientNetB2 v3
classifier = models.efficientnet_b2(weights=None)
classifier.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(classifier.classifier[1].in_features, 4)
)

checkpoint = torch.load("classifier_best.pth", map_location=device)
if "model_state_dict" in checkpoint:
    classifier.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded v2 — best val acc: {checkpoint.get('best_val_acc', 'N/A')}")
else:
    classifier.load_state_dict(checkpoint)

classifier.eval().to(device)
print("Model ready!")

In [ ]:
# NOTE: exact duplicate of the "10.3 Gradcam Test" cell above (cell-41) — kept as-is per the
# conservative editing policy for this documentation pass rather than deleted; see Current Status.
#!pip install grad-cam -q
import torchvision.transforms as T
import numpy as np
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

print("Upload your hard test images")
uploaded_imgs = files.upload()
image_paths = list(uploaded_imgs.keys())

target_layer = [model.features[-1]]
transform_test = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

CLASSES = list(train_data.classes)
n = len(image_paths)
fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
if n == 1: axes = [axes]

for i, path in enumerate(image_paths):
    img_pil = Image.open(path).convert("RGB").resize((224, 224))
    img_np  = np.array(img_pil).astype(np.float32) / 255.0
    tensor  = transform_test(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)[0]

    pred_idx   = probs.argmax().item()
    pred_brand = CLASSES[pred_idx]
    confidence = round(probs[pred_idx].item() * 100, 1)

    with GradCAM(model=model, target_layers=target_layer) as cam:
        grayscale = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred_idx)])
        cam_image = show_cam_on_image(img_np, grayscale[0], use_rgb=True)

    axes[i][0].imshow(img_np)
    axes[i][0].set_title(f"Original\n{path}", fontsize=10)
    axes[i][0].axis("off")

    axes[i][1].imshow(cam_image)
    axes[i][1].set_title(f"GradCAM\n{pred_brand} ({confidence}%)", fontsize=10)
    axes[i][1].axis("off")

    probs_list = [round(probs[j].item()*100, 1) for j in range(len(CLASSES))]
    colors = ["#e74c3c", "#3498db", "#2ecc71", "#95a5a6"][:len(CLASSES)]
    bars = axes[i][2].barh(CLASSES, probs_list, color=colors)
    axes[i][2].set_xlim(0, 110)
    axes[i][2].set_xlabel("Confidence (%)")
    axes[i][2].set_title("Probabilities")
    for bar, val in zip(bars, probs_list):
        axes[i][2].text(val+1, bar.get_y()+bar.get_height()/2,
                        f"{val}%", va="center", fontsize=10, fontweight="bold")
    axes[i][2].spines[["top","right"]].set_visible(False)

plt.suptitle("GradCAM — EfficientNetB2 v3", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("gradcam_v3.png", dpi=150, bbox_inches="tight")
plt.show()
files.download("gradcam_v3.png")